## Generating Ground Truth Data

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [2]:
model = "qwen3.5-9b_coding_fast"

In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [6]:
# assign llm course to documents var
documents = documents_llm

In [7]:
len(documents)

118

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.

Use casual language
""".strip()

In [10]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.chat.completions.parse(
    model=model,
    messages=messages,
    response_format=Questions
)

In [15]:
response

ParsedChatCompletion[TypeVar](id='chatcmpl-CQLVAxJt6DnU66DxXZ2tKdlVgj7sshXE', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{\n  "questions": [\n    "Is there still a way to get into this LLM Zoomcamp after the launch?",\n    "Will I still be able to earn a cert if I start now?",\n    "Do I have to hand in my assignment before the intake closes?",\n    "Does starting late mean I miss out on the official badge?",\n    "Is there a cutoff for finishing the work if I want the badge?"\n  ]\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, parsed=Questions(questions=['Is there still a way to get into this LLM Zoomcamp after the launch?', 'Will I still be able to earn a cert if I start now?', 'Do I have to hand in my assignment before the intake closes?', 'Does starting late mean I miss out on the official badge?', 'Is there a cutoff for finishing the work

In [16]:
results = response.choices[0].message.parsed
results

Questions(questions=['Is there still a way to get into this LLM Zoomcamp after the launch?', 'Will I still be able to earn a cert if I start now?', 'Do I have to hand in my assignment before the intake closes?', 'Does starting late mean I miss out on the official badge?', 'Is there a cutoff for finishing the work if I want the badge?'])

In [17]:
results.questions

['Is there still a way to get into this LLM Zoomcamp after the launch?',
 'Will I still be able to earn a cert if I start now?',
 'Do I have to hand in my assignment before the intake closes?',
 'Does starting late mean I miss out on the official badge?',
 'Is there a cutoff for finishing the work if I want the badge?']

In [18]:
usage_dict = {
    "prompt_tokens": response.usage.prompt_tokens,
    "total_tokens": response.usage.total_tokens
}

In [19]:
usage_dict['prompt_tokens'], usage_dict['total_tokens']

(189, 5538)

In [24]:
from evaluation_utils import llm_structured_chat_completions, calc_price

In [21]:
result, usage = llm_structured_chat_completions(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [26]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it actually too late to get on board right now?',
  'document': '74eb249bbf'},
 {'question': 'Will starting late mean I miss out on the badge?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish the final task before a specific deadline?',
  'document': '74eb249bbf'},
 {'question': 'Can new people participate even after the initial sign up period has passed?',
  'document': '74eb249bbf'},
 {'question': 'How does being enrolled late affect getting the award?',
  'document': '74eb249bbf'}]